In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm.auto import tqdm
from torchvision import models
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import copy

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED = 42

MODELS_DIR = Path("../models")
ARTIFACTS_DIR = Path("../reports")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = MODELS_DIR / "best_resnet18.pth"

g = torch.Generator().manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
trainval_ds = datasets.OxfordIIITPet(
    root=str(DATA_DIR),
    split="trainval",
    target_types="category",
    download=True,
    transform=train_tfms,
)

test_ds = datasets.OxfordIIITPet(
    root=str(DATA_DIR),
    split="test",
    target_types="category",
    download=True,
    transform=eval_tfms,
)

len(trainval_ds), len(test_ds)

In [ ]:
val_ratio = 0.2
n_total = len(trainval_ds)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_ds, val_ds = random_split(trainval_ds, [n_train, n_val], generator=g)

len(train_ds), len(val_ds)

In [ ]:
trainval_ds_eval = datasets.OxfordIIITPet(
    root=str(DATA_DIR),
    split="trainval",
    target_types="category",
    download=False,
    transform=eval_tfms,
)

val_ds.dataset = trainval_ds_eval

In [ ]:
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

(len(train_loader), len(val_loader), len(test_loader))

In [ ]:
images, labels = next(iter(train_loader))
images.shape, labels.shape, labels[:10]

In [ ]:
class_names = trainval_ds.classes
num_classes = len(class_names)
num_classes, class_names[:5]

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model = model.to(device)
model

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

In [ ]:
images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    logits = model(images)

logits.shape

In [ ]:
def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == y).sum().item()
    return correct / y.size(0)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    pbar = tqdm(loader, desc="train", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == labels).sum().item()
        running_total += batch_size

        pbar.set_postfix(loss=loss.item())

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    pbar = tqdm(loader, desc="val", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == labels).sum().item()
        running_total += batch_size

        pbar.set_postfix(loss=loss.item())

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc

In [ ]:
def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs=5, ckpt_path=None, scheduler=None):
    best_val_acc = 0.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        print(
            f"Epoch {epoch:>2}/{epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

            if ckpt_path is not None:
                torch.save(
                    {
                        "model_state_dict": best_state,
                        "val_acc": best_val_acc,
                        "epoch": epoch,
                    },
                    ckpt_path,
                )
                print(f"saved checkpoint -> {ckpt_path} (val_acc={best_val_acc:.4f})")

        if scheduler is not None:
            scheduler.step()

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"Best val acc: {best_val_acc:.4f}")
    return history

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=5,
    ckpt_path=CKPT_PATH,
)

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss = [h["val_loss"] for h in history]
train_acc = [h["train_acc"] for h in history]
val_acc = [h["val_acc"] for h in history]

outpath = ARTIFACTS_DIR / "training_curves"

plt.figure()
plt.plot(epochs, train_loss, label="train_loss")
plt.plot(epochs, val_loss, label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(outpath.with_name(outpath.stem + "_loss.png"), dpi=200, bbox_inches="tight")
plt.show()

plt.figure()
plt.plot(epochs, train_acc, label="train_acc")
plt.plot(epochs, val_acc, label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(outpath.with_name(outpath.stem + "_acc.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
def plot_and_save_training_curves(history, outpath: Path):
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    train_acc = [h["train_acc"] for h in history]
    val_acc = [h["val_acc"] for h in history]

    plt.figure()
    plt.plot(epochs, train_loss, label="train_loss")
    plt.plot(epochs, val_loss, label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath.with_name(outpath.stem + "_loss.png"), dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure()
    plt.plot(epochs, train_acc, label="train_acc")
    plt.plot(epochs, val_acc, label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath.with_name(outpath.stem + "_acc.png"), dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device, return_images=False):
    model.eval()
    all_true = []
    all_pred = []
    all_images = [] if return_images else None

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        preds = logits.argmax(dim=1)

        all_true.append(labels.cpu().numpy())
        all_pred.append(preds.cpu().numpy())

        if return_images:
            all_images.append(images.cpu())
    
    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    if return_images:
        imgs = torch.cat(all_images, dim=0)
        return y_true, y_pred, imgs
    
    return y_true, y_pred

In [ ]:
y_true_val, y_pred_val, val_images = collect_predictions(model, val_loader, device, return_images=True)

In [ ]:
def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def plot_and_save_confusion_matrix(cm, class_names, outpath, normalize=True, max_classes_to_show=37):
    cm_plot = cm.astype(np.float64)
    if normalize:
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        cm_plot = np.divide(cm_plot, row_sums, where=row_sums != 0)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title("Confusion Matrix" + (" (normalized)" if normalize else ""))
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=90, fontsize=7)
    plt.yticks(ticks, class_names, fontsize=7)
    plt.tight_layout()
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.show()

cm_val = confusion_matrix_np(y_true_val, y_pred_val, num_classes)
cm_path = ARTIFACTS_DIR / "confusion_matrix_val.png"
plot_and_save_confusion_matrix(cm_val, class_names, cm_path, normalize=True)

In [ ]:
def per_class_accuracy(cm):
    correct = np.diag(cm)
    total = cm.sum(axis=1)
    acc = np.divide(correct, total, where=total != 0)
    return acc, total

acc, support = per_class_accuracy(cm_val)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "class": class_names,
    "accuracy": acc,
    "support": support,
}).sort_values("accuracy")

csv_path = ARTIFACTS_DIR / "per_class_accuracy_val.csv"
df.to_csv(csv_path, index=False)
df.head(10)

In [ ]:
def unnormalize(img_batch):
    return img_batch * IMAGENET_STD + IMAGENET_MEAN

def show_error_gallery(images_norm, y_true, y_pred, class_names, n=16):
    wrong_idx = np.where(y_true != y_pred)[0]
    if len(wrong_idx) == 0:
        print("No misclassifications found!")
        return

    chosen = wrong_idx[:n]
    imgs = images_norm[chosen]

    imgs = unnormalize(imgs).clamp(0, 1)

    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(12, 3*rows))

    for i, idx in enumerate(chosen):
        plt.subplot(rows, cols, i+1)
        img = imgs[i].permute(1,2,0).numpy()
        plt.imshow(img)
        t = class_names[y_true[idx]]
        p = class_names[y_pred[idx]]
        plt.title(f"T: {t}\nP: {p}", fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_error_gallery(val_images, y_true_val, y_pred_val, class_names, n=16)

In [ ]:
for p in model.parameters():
    p.requires_grad = False

for p in model.layer4.parameters():
    p.requires_grad = True

for p in model.fc.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    [
        {"params": model.layer4.parameters(), "lr": 1e-4},
        {"params": model.fc.parameters(), "lr": 1e-3},
    ],
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [ ]:
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name)

In [ ]:
history_ft = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=5,
    ckpt_path=MODELS_DIR / "best_resnet18_layer4.pth",
    scheduler=scheduler,
)

In [ ]:
ckpt_path = MODELS_DIR / "best_resnet18_layer4.pth"
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded checkpoint from epoch:", ckpt.get("epoch"), "val_acc:", ckpt.get("val_acc"))

In [ ]:
y_true_test, y_pred_test = collect_predictions(model, test_loader, device, return_images=False)

test_acc = (y_true_test == y_pred_test).mean()
print(f"Final TEST accuracy: {test_acc:.4f}")

test_loss, _ = evaluate(model, test_loader, criterion, device)
print(f"Final TEST loss: {test_loss:.4f}")

In [ ]:
cm_test = confusion_matrix_np(y_true_test, y_pred_test, num_classes)
plot_and_save_confusion_matrix(
    cm_test, class_names, ARTIFACTS_DIR / "confusion_matrix_test.png", normalize=True
)

acc_test, support_test = per_class_accuracy(cm_test)

pd.DataFrame({
    "class": class_names,
    "accuracy": acc_test,
    "support": support_test
}).sort_values("accuracy").to_csv(ARTIFACTS_DIR / "per_class_accuracy_test.csv", index=False)

In [ ]:
plot_and_save_training_curves(history, ARTIFACTS_DIR / "curves_baseline")
plot_and_save_training_curves(history_ft, ARTIFACTS_DIR / "curves_layer4_ft")